# Framing dan checksum

Link serial butuh dua hal: cara menemukan batas frame (SLIP, COBS, HDLC) dan cara mendeteksi kerusakan (CRC). IoTCom.Net.Framing menyediakan keduanya, tanpa alokasi dan streaming.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.4.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.4.0-preview.1"

## Katalog CRC
Setiap preset diverifikasi dengan nilai check katalog untuk ASCII `123456789`.

In [ ]:
using IoTCom.Net.Framing;
foreach (var crc in CrcCatalog.All)
    Console.WriteLine($"{crc.Parameters.Name,-18} 0x{crc.Compute("123456789"u8):X}  ok={crc.SelfTest()}");

## Byte yang sama, tiga framing

In [ ]:
using System.Buffers;
byte[] payload = [0x7E, 0xC0, 0x00, 0x11, 0xDB];
foreach (IFrameEncoder codec in new IFrameEncoder[] { new Slip(), new Cobs(), new Hdlc() })
{
    var w = new ArrayBufferWriter<byte>();
    codec.Encode(payload, w);
    Console.WriteLine($"{codec.GetType().Name,-5} {IoTCom.Net.HexDump.ToHex(w.WrittenSpan)}");
}

## Decode streaming
Decoder bekerja pada `ReadOnlySequence<byte>` langsung dari `PipeReader`, sehingga frame yang terpotong di beberapa read tetap tertangani.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*

In [ ]:
using System.IO.Pipelines;
var pipe = new Pipe();
var cobs = new Cobs();
await pipe.Writer.WriteFrameAsync(cobs, new byte[] { 1, 0, 2 });
await pipe.Writer.WriteFrameAsync(cobs, new byte[] { 3, 4 });
await pipe.Writer.CompleteAsync();
await foreach (var frame in pipe.Reader.ReadFramesAsync(cobs)) Console.WriteLine(IoTCom.Net.HexDump.ToHex(frame));